In [1]:
import pathlib

import numpy as np
import pandas as pd
import pyarrow as pa
import swifter  # noqa: F401

from pyarrow import parquet as pq
from rdkit import Chem
from tqdm.auto import tqdm

In [2]:
output_file = "species_semiempirical_results.parquet"

In [3]:
cwd = pathlib.Path().absolute()
quantum_green = pathlib.Path("/home/shared/projects/quantum_green")
paquets = quantum_green / "supercloud_xfer"
data_dir = cwd.parent.parent / "data" / "qm_results"

In [4]:
pd.set_option("display.max_columns", None)


def head(df, n=2):
    display(df.head(n))
    print(f"Contains {len(df)} rows")


def swifter_apply(seris, func, desc="Applying function"):
    return seris.swifter.progress_bar(True, desc=desc).apply(func)


def extract_last(lst):
    if isinstance(lst, list) and len(lst) > 0:
        return lst[-1]
    return lst

In [5]:
species_data = pd.read_pickle(
    quantum_green / "reactants" / "quantum_green_species_data_24march12b.pkl"
)
head(species_data)

,asmi,species_id,species_dft_log_path,species_dft_route_section,charge,multiplicity,species_dft_opt_max_steps,species_dft_opt_normal_termination_check,species_dft_opt_cpu_time,species_dft_opt_wall_time,species_dft_sum_of_electronic_and_thermal_enthalpies_hartree,species_dft_hartreefock_energy_hartree,species_dft_zpe_hartree,species_dft_e0_zpe_hartree,species_dft_gibbs_hartree,species_dft_scf_hartree,species_dft_frequency_modes,species_dft_std_forces,species_dft_opted_std_xyz,species_dft_opted_xyz_input_orientation,is_ts,batch_label,species_dft_log_filename_id,nasmi,matched_rxn_fingerprint,species_dft_log_source_utf_8_sha512,passed_connectivity_check,rdkit_canonical_nasmi,partent_rdkit_canonical_nasmi_radical_only,reaction_center_atom_map_num_radical_only,std_xyz_str,xyz_str,matched_parent_mol_sha512_radical_only,matched_parent_mol_asmi_radical_only,matched_parent_mol_reaction_center_atom_map_num_radical_only,species_dft_first_freq,species_dlpno_log_filename_id,species_dlpno_log_path,species_dlpno_sp_hartree,dlpno_batch_label,species_dft_frequencies
0,[C:1]([C:2]1([H:13])[C:3]([H:14])([H:15])[N:4]...,1,/home/gridsan/groups/RMG/Projects/Hao-Wei-Osca...,"P opt=(calcfc,maxcycle=128,noeig,nomicro,carte...",0,2,102,True,4100.1,1076.5,-434.175602,-434.321489,0.136748,-434.184741,-434.218451,434.321490,"[[[1.0, 6.0, -0.01, 0.18, 0.17], [2.0, 6.0, -0...","[[1.0, 6.0, 1.274e-05, 2.05e-06, 3.763e-06], [...","[[1.0, 6.0, 0.0, 1.687648, -1.481461, 0.769379...","[[1.0, 6.0, 0.0, 1.711519, -1.472917, 0.740063...",False,aug11b,id52810,CC1C[N]C(=N)N1C=O,"{('46e86ef9080f8c5b21f5c329d23f0d6c', 'p2_smi')}",42d67c486eb53d22486a5d1da768e644bf644469198894...,True,CC1C[N]C(=N)N1C=O,CC1CNC(=N)N1C=O,4.0,17\n\nC 1.687648 -1.481461 0.769379\nC 1.14635...,17\n\nC 1.711519 -1.472917 0.740063\nC 1.17193...,4f388644fa3d52bec962545fb1a65c4446aa8758d8834c...,[C:1]([C:2]1([H:13])[C:3]([H:14])([H:15])[N:4]...,16,69.0965,id52810,/home/gridsan/jburns/RMG_shared/Projects/Hao-W...,-434.224944,reactant_aug11b,"[69.0965, 133.1139, 185.0048, 247.5052, 269.23..."
1,[C:1]([C:2]1([H:13])[C:3]([H:14])([H:15])[O:4]...,3,/home/gridsan/groups/RMG/Projects/Hao-Wei-Osca...,"P opt=(calcfc,maxcycle=128,noeig,nomicro,carte...",0,2,126,True,12128.7,3218.9,-403.235873,-403.429122,0.183595,-403.245527,-403.280167,403.429122,"[[[1.0, 6.0, 0.21, 0.06, 0.18], [2.0, 6.0, 0.0...","[[1.0, 6.0, 5.513e-06, 2.7033e-05, 2.003e-06],...","[[1.0, 6.0, 0.0, -2.800669, 0.461226, 0.774726...","[[1.0, 6.0, 0.0, 2.701051, 1.034704, 0.04014],...",False,aug11b,id52729,CC1COC(C2C[N]2)C1,"{('5ddcee1ad82096fe7dcfd17162f30b09', 'p2_smi')}",7667c6f680d9065479d515666e4c3246f65704e4926140...,True,CC1COC(C2C[N]2)C1,CC1COC(C2CN2)C1,8.0,21\n\nC -2.800669 0.461226 0.774726\nC -1.7902...,21\n\nC 2.701051 1.034704 0.04014\nC 1.73068 -...,f0d1bccdf82215b9726f7ea9b5b912db1d1b0a4b8fd7e0...,[C:1]([C:2]1([H:13])[C:3]([H:14])([H:15])[O:4]...,20,40.2091,id52729,/home/gridsan/jburns/RMG_shared/Projects/Hao-W...,-403.298238,reactant_aug11b,"[40.2091, 100.6031, 169.247, 232.8355, 250.021..."


Contains 348258 rows


In [6]:
matching_df = pd.read_csv(cwd / "quantum_green_species_data_match.csv")
matching_df = (
    matching_df[matching_df["semiempirical_index"] >= 0]
    .sort_values("semiempirical_index")
    .reset_index(drop=True)
    .copy()
)
matching_df["smiles"] = matching_df["species_id"].map(
    species_data.set_index("species_id")["asmi"]
)
head(matching_df)

,species_id,dft_index,dlpno_index,semiempirical_index,rmsd,from_initial,smiles
0,2391,2411,169275,0,0.0,False,[C:1]([C:2]1([H:13])[O:3][C:4]2([H:14])[C:5]3(...
1,1630,789,169859,6,0.0,False,[C:1]([C:2]1([H:13])[C:3]([H:14])([H:15])[C:4]...


Contains 348190 rows


In [7]:
def read_column(name):
    return pq.read_table(
        paquets / "semiempirical_nonts_semiempirical_v9.parquet",
        columns=[name],
    )[name]

In [8]:
normal_termination = read_column("normal_termination")
keep_index = np.zeros(len(normal_termination), dtype=bool)
keep_index[matching_df["semiempirical_index"].values] = True
keep_index = np.logical_and(keep_index, normal_termination.to_pylist())
row_filter = pa.array(keep_index)
sum(keep_index).item(), row_filter.sum().as_py()

(348190, 348190)

In [9]:
column_metadata = {
    "smiles": "Species SMILES (with atom mapping indicating xyz and force order)",
    "route_section": "Level of theory",
    "charge": "Molecular formal charge",
    "multiplicity": "Electron multiplicity",
    "max_steps": "Maximum allowed steps",
    "cpu_time": "CPU time in seconds",
    "wall_time": "Wall time in seconds",
    "e0_h": "Enthalpy at 298K",
    "hf": "E0 for non-wavefunction methods",
    "zpe_per_atom": "Per-atom zero point energy",
    "e0_zpe": "Gibbs free energy at 0K",
    "gibbs": "Gibbs free energy at 298K",
    "frequencies": "Vibrational frequencies",
    "frequency_modes": "Vibrational modes",
    "initial_xyz": "Input XYZ coords",
    "std_xyz": "Standardized XYZ coords after optimization",
    "std_forces": "Standardized forces after optimization",
}

In [10]:
semiempirical_data = pa.Table.from_arrays(
    [pa.array(matching_df["smiles"])],
    schema=pa.schema([pa.field("smiles", pa.string())], metadata=column_metadata),
)

pbar = tqdm([key for key in column_metadata.keys() if key != "smiles"])
for name in pbar:
    pbar.set_description(f"Processing column {name}")
    column = read_column(name).filter(row_filter)
    if name in ["std_xyz", "std_forces"]:
        column = pa.array([extract_last(item.as_py()) for item in column])
    old_semiempirical_data = semiempirical_data
    semiempirical_data = old_semiempirical_data.append_column(name, column)
    del old_semiempirical_data
    del column

  0%|          | 0/16 [00:00<?, ?it/s]

In [11]:
semiempirical_data.schema

smiles: string
route_section: string
charge: uint8
multiplicity: uint8
max_steps: uint32
cpu_time: uint32
wall_time: uint32
e0_h: double
hf: double
zpe_per_atom: double
e0_zpe: double
gibbs: double
frequencies: list<element: double>
  child 0, element: double
frequency_modes: list<element: list<element: list<element: double>>>
  child 0, element: list<element: list<element: double>>
      child 0, element: list<element: double>
          child 0, element: double
initial_xyz: list<element: list<element: float>>
  child 0, element: list<element: float>
      child 0, element: float
std_xyz: list<item: list<item: double>>
  child 0, item: list<item: double>
      child 0, item: double
std_forces: null
-- schema metadata --
smiles: 'Species SMILES (with atom mapping indicating xyz and force order' + 1
route_section: 'Level of theory'
charge: 'Molecular formal charge'
multiplicity: 'Electron multiplicity'
max_steps: 'Maximum allowed steps'
cpu_time: 'CPU time in seconds'
wall_time: 'Wall ti

In [12]:
semiempirical_df = semiempirical_data.to_pandas()
head(semiempirical_df)

,smiles,route_section,charge,multiplicity,max_steps,cpu_time,wall_time,e0_h,hf,zpe_per_atom,e0_zpe,gibbs,frequencies,frequency_modes,initial_xyz,std_xyz,std_forces
0,[C:1]([C:2]1([H:13])[O:3][C:4]2([H:14])[C:5]3(...,"opt=(calcall,maxcycle=128,noeig,nomicro,cartes...",0,1,114,16,5,-27.989950,-28.155358,0.156461,-27.998897,-28.031254,"[126.8161, 132.6914, 177.2304, 208.9232, 252.2...","[[[1.0, 6.0, -0.02, 0.08, 0.02], [2.0, 6.0, 0....","[[1.0, 6.0, 0.0, 2.5503, -0.2105, 0.3812], [2....","[[1.0, 6.0, 0.0, 2.6694300174713135, -0.034522...",None
1,[C:1]([C:2]1([H:13])[C:3]([H:14])([H:15])[C:4]...,"opt=(calcall,maxcycle=128,noeig,nomicro,cartes...",0,2,108,53,18,0.045825,-0.102202,0.137866,0.035664,-0.000526,"[34.9384, 52.5418, 107.9471, 144.6069, 180.651...","[[[1.0, 6.0, -0.11, -0.08, -0.2], [2.0, 6.0, 0...","[[1.0, 6.0, 0.0, -1.7136, 0.9969, -0.3624], [2...","[[1.0, 6.0, 0.0, -1.5091789960861206, 1.817577...",None


Contains 348190 rows


In [13]:
rows_to_delete = set()
for col in semiempirical_df.columns:
    if col == "std_forces":
        continue
    to_delete = semiempirical_df.index[semiempirical_df[col].isna()]
    rows_to_delete.update(to_delete)
    if len(to_delete) > 0:
        print(f"Column {col} has {len(to_delete)} missing values")

print(f"\nMarked {len(rows_to_delete)} rows for deletion")

Column frequency_modes has 1 missing values

Marked 1 rows for deletion


In [14]:
energy_cols = ["e0_h", "hf", "e0_zpe", "gibbs"]
mean_energy = semiempirical_df[energy_cols].mean(axis=1)
for col in energy_cols:
    deviations = (semiempirical_df[col] - mean_energy).abs()
    rows_to_delete.update(deviations.index[deviations > 1.0])

print(f"\nA total of {len(rows_to_delete)} rows are now marked for deletion.")


A total of 1 rows are now marked for deletion.


In [15]:
PARAMS = Chem.SmilesParserParams()
PARAMS.removeHs = False


def get_elements_from_smiles(smiles):
    reactant_smiles = smiles.split(">>")[0].split(".")
    mols = [Chem.MolFromSmiles(smi, PARAMS) for smi in reactant_smiles]
    order = {
        atom.GetAtomMapNum(): atom.GetSymbol()
        for mol in mols
        for atom in mol.GetAtoms()
    }
    try:
        return "".join([order[i] for i in range(1, len(order) + 1)])
    except TypeError:
        return ""


PERIODIC_TABLE = Chem.GetPeriodicTable()


def get_elements_from_xyz(xyz):
    if xyz is None:
        return None
    return "".join(PERIODIC_TABLE.GetElementSymbol(int(item[1])) for item in xyz)

In [16]:
elements_from_xyz = swifter_apply(
    semiempirical_df["std_xyz"], get_elements_from_xyz, "Getting elements from xyz"
)
elements_from_smiles = swifter_apply(
    semiempirical_df["smiles"],
    get_elements_from_smiles,
    "Getting elements from smiles",
)
rows_to_delete |= set(semiempirical_df.index[elements_from_smiles != elements_from_xyz])

print(f"\nA total of {len(rows_to_delete)} rows are now marked for deletion.")

Getting elements from xyz:   0%|          | 0/348190 [00:00<?, ?it/s]

Getting elements from smiles:   0%|          | 0/348190 [00:00<?, ?it/s]


A total of 1 rows are now marked for deletion.


In [17]:
keep_index = np.ones(len(semiempirical_df), dtype=bool)
keep_index[np.array(list(rows_to_delete), dtype=int)] = False
filtered_semiempirical_data = semiempirical_data.filter(pa.array(keep_index))
del semiempirical_data

In [18]:
data_dir.mkdir(parents=True, exist_ok=True)
pq.write_table(filtered_semiempirical_data, data_dir / output_file, compression="zstd")

In [19]:
parquet_file = pq.ParquetFile(data_dir / output_file)
print(f"Successfully wrote parquet file with {parquet_file.metadata.num_rows} rows")

Successfully wrote parquet file with 348189 rows
